# Airbnb Paris — Prédiction du prix par nuitée

## Plan du notebook
1. Chargement des données
2. Exploration & nettoyage
3. Feature engineering (amenities, parsing price)
4. Préparation du pipeline (ColumnTransformer)
5. Modélisation & comparaison (CV 5-fold)
6. Analyse des résidus
7. Conclusion

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import load_raw_data, parse_price, parse_amenities, create_amenity_features
from src.modeling import build_preprocessor, get_models, build_pipeline, evaluate_model

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

%matplotlib inline

## 1. Chargement des données

In [ ]:
df = load_raw_data("../data/raw/listings.csv")
print(f"Shape: {df.shape}")
df.head()

## 2. Exploration & Nettoyage

In [ ]:
# Aperçu des types et valeurs manquantes
df.info()

In [ ]:
# Parsing du prix
df["price"] = parse_price(df["price"])

# Suppression des prix à 0 ou aberrants
df = df[(df["price"] > 0) & (df["price"] < 10000)]
print(f"Shape après filtrage prix: {df.shape}")

# Distribution du prix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["price"], bins=100)
axes[0].set_title("Distribution du prix (brut)")
axes[1].hist(np.log1p(df["price"]), bins=100)
axes[1].set_title("Distribution du prix (log1p)")
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Parsing des amenities
parsed_amenities, all_amenities = parse_amenities(df["amenities"])
print(f"Nombre total d'amenities uniques: {len(all_amenities)}")

# Sélection des amenities les plus pertinentes (au moins 5)
selected_amenities = ["Wifi", "TV", "Elevator", "Washer", "Dryer", "Air conditioning", "Balcony"]

amenity_df = create_amenity_features(parsed_amenities, selected_amenities)
amenity_df.head()

In [ ]:
# Sélection des features
numeric_features = ["accommodates", "bedrooms", "beds", "bathrooms"]
categorical_features = ["neighbourhood_cleansed", "room_type", "property_type"]

# Construction du DataFrame final
# TODO: adapter selon les colonnes disponibles dans votre version du dataset
df_model = df[numeric_features + categorical_features + ["price"]].copy()
df_model = pd.concat([df_model.reset_index(drop=True), amenity_df.reset_index(drop=True)], axis=1)

# Gestion des NaN
df_model[numeric_features] = df_model[numeric_features].fillna(df_model[numeric_features].median())
df_model[categorical_features] = df_model[categorical_features].fillna("Unknown")
df_model = df_model.dropna(subset=["price"])

print(f"Shape finale: {df_model.shape}")
df_model.head()

## 4. Pipeline & Modélisation

In [ ]:
# Séparation X / y
amenity_cols = [c for c in df_model.columns if c.startswith("has_")]
feature_cols = numeric_features + categorical_features + amenity_cols

X = df_model[feature_cols]
y = df_model["price"]

print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
# Construction du preprocessor
all_numeric = numeric_features + amenity_cols
preprocessor = build_preprocessor(all_numeric, categorical_features)

# Comparaison des modèles
models = get_models()
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = build_pipeline(preprocessor, model)
    scores = evaluate_model(pipeline, X, y, cv=5)
    results[name] = scores
    print(f"  R² = {scores['r2_mean']:.4f} ± {scores['r2_std']:.4f}")
    print(f"  MAE = {scores['mae_mean']:.2f}€ ± {scores['mae_std']:.2f}€")
    print()

In [ ]:
# Tableau récapitulatif
results_df = pd.DataFrame(results).T
results_df.columns = ["R² (mean)", "R² (std)", "MAE (mean)", "MAE (std)"]
results_df.sort_values("R² (mean)", ascending=False)

## 5. Analyse des résidus (meilleur modèle)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le meilleur modèle sur le train set
best_model_name = results_df["R² (mean)"].idxmax()
print(f"Meilleur modèle: {best_model_name}")

best_pipeline = build_pipeline(preprocessor, get_models()[best_model_name])
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

# Résidus
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Prédiction (€)")
axes[0].set_ylabel("Résidu (€)")
axes[0].set_title("Résidus vs Prédictions")

axes[1].hist(residuals, bins=100)
axes[1].set_title("Distribution des résidus")

axes[2].scatter(y_test, y_pred, alpha=0.3, s=5)
axes[2].plot([0, y_test.max()], [0, y_test.max()], "r--")
axes[2].set_xlabel("Prix réel (€)")
axes[2].set_ylabel("Prix prédit (€)")
axes[2].set_title("Réel vs Prédit")

plt.tight_layout()
plt.show()

## 6. Conclusion

- **Meilleur modèle** : à compléter après exécution
- **MAE** : à compléter (en €)
- **Points d'amélioration** : 
  - Plus de features textuelles (NLP sur description)
  - Tuning des hyperparamètres
  - Features géographiques (latitude/longitude, distance aux monuments)